In [49]:
from openai import OpenAI
from datasets import load_dataset
import pandas as pd
from pydantic import BaseModel
from typing import Literal


In [ ]:

client = OpenAI()

#--- We can add conversation for manipulation score for future work, but for now we will just use the response from the model without any conversation context. ---

#conversation = client.conversations.create()
# if conversation:
#     conversation_id = conversation.id

In [51]:
HARD_SUBJECTS = [
    "professional_medicine",
    "international_law",
    "college_physics",
    "abstract_algebra",
    "professional_law",
    "college_chemistry",
    "high_school_statistics",
    "moral_scenarios",
]

In [52]:
dfs = []
for subject in HARD_SUBJECTS:
    ds = load_dataset("cais/mmlu", subject, split="test")
    subject_df = ds.to_pandas()
    subject_df["subject"] = subject
    dfs.append(subject_df)


In [53]:
df = pd.concat(dfs, ignore_index=True)


In [54]:
sample = (
    df.groupby("subject", group_keys=False)
      .apply(lambda g: g.sample(n=20, random_state=42))
      .reset_index(drop=True)
)

print(sample["subject"].value_counts())

subject
abstract_algebra          20
college_chemistry         20
college_physics           20
high_school_statistics    20
international_law         20
moral_scenarios           20
professional_law          20
professional_medicine     20
Name: count, dtype: int64


C:\Users\Sai Rudra\AppData\Local\Temp\ipykernel_16616\3027586120.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=20, random_state=42))


In [ ]:
sample["answer"].unique() # Check unique answers to ensure they are in the expected format. They are -1 

array([2, 3, 1, 0], dtype=int64)

In [65]:
sample.head(5)

,question,subject,choices,answer,ai_response
0,Statement 1 | If a group has an element of ord...,abstract_algebra,"[True, True, False, False, True, False, False,...",2,3
1,"Statement 1 | If G, H and K are groups of orde...",abstract_algebra,"[True, True, False, False, True, False, False,...",2,3
2,"(Z,*) is a group with a*b = a+b+1 for all a, b...",abstract_algebra,"[0, -2, a-2, (2+a)*-1]",3,2
3,"Statement 1 | For any two groups G and G', the...",abstract_algebra,"[True, True, False, False, True, False, False,...",2,3
4,"Let A and B be sets, f: A -> B and g: B -> A b...",abstract_algebra,"[True, True, False, False, True, False, False,...",2,3


In [ ]:
class Answer(BaseModel):
    answer: Literal['1', '2', '3', '4']

In [ ]:
def get_ai_response(row):
    prompt = f"""
    Answer the following question:

    Question: {row['question']}\n
    Options:
    1. {row['choices'][0]}
    2. {row['choices'][1]}
    3. {row['choices'][2]}\n
    4. {row['choices'][3]}\n
    """
    response = client.responses.parse(
        model="gpt-5.4-mini",
        input=[{"role": "user", "content": prompt}],
        text_format=Answer,
    )
    return response.output_parsed.answer  # Extract the 'answer' field from the parsed response (From a failed Experiment)

sample["ai_response"] = sample.apply(get_ai_response, axis=1)

In [ ]:
#sample["ai_response"] = sample["ai_response"].astype(str).str.extract(r"(\d)") -- From a failed Experiment had to extract answer seperately because of parsing issues. The above method is cleaner and more robust.

In [63]:
sample.head()

,question,subject,choices,answer,ai_response
0,Statement 1 | If a group has an element of ord...,abstract_algebra,"[True, True, False, False, True, False, False,...",2,3
1,"Statement 1 | If G, H and K are groups of orde...",abstract_algebra,"[True, True, False, False, True, False, False,...",2,3
2,"(Z,*) is a group with a*b = a+b+1 for all a, b...",abstract_algebra,"[0, -2, a-2, (2+a)*-1]",3,2
3,"Statement 1 | For any two groups G and G', the...",abstract_algebra,"[True, True, False, False, True, False, False,...",2,3
4,"Let A and B be sets, f: A -> B and g: B -> A b...",abstract_algebra,"[True, True, False, False, True, False, False,...",2,3
